# SOP-UQ Quick Start

Use your **trial token** and run the cells from top to bottom. You will:

1. connect to your team's examiner and to the Qwen model service;
2. look at the data and call business tools and the oracle by hand;
3. run the baseline agent, collect labeled trajectories and fit risk estimators;
4. Problem 1: estimate the risks of the 360 test cases and submit them;
5. Problem 2: read oracle facts on the interactive cases, decide and submit.

For environment setup and connection settings, start with the [README](../README.md).
You need five values from the examiner: examiner URL, your team's trial token and
formal token, model URL and model token. Trial submissions are checked but not
scored; trial tool calls and oracle reads have a small quota.

Sections 6 and 7 each submit a CSV. Using the formal token for the example
submissions uses your scored tries. **For formal runs, follow the stages below
rather than Run All:** finish your Problem 1 improvements and submissions before
starting Problem 2. The [candidate manual](../docs/candidate_manual.pdf) introduces
the project and explains the rules; the [problem statement](../docs/problem_statement.pdf)
defines the task.


In [ ]:
from pathlib import Path
import json, os, sys

# Works whether Jupyter starts in the project folder or in scripts/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "test_client").is_dir())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
WORK = ROOT / ".runs" / "notebook"          # all outputs of this notebook (git-ignored)

# Your examiner and model connections. Set them as environment variables before
# starting Jupyter, or type them here (do not share notebooks containing tokens).
EXAMINER_URL = os.environ.get("SOP_EXAMINER_URL", "http://127.0.0.1:8801")
EXAMINER_TOKEN = os.environ.get("SOP_EXAMINER_TOKEN", "")
MODEL_URL = os.environ.get("SOP_MODEL_URL", "http://127.0.0.1:18080")
MODEL_TOKEN = os.environ.get("SOP_MODEL_TOKEN", "")

## 1. Connect

`spec()` shows which token you use (`trial` or `exam`), the costs, the per-case
limits, which of the 360 test cases are interactive (tools and oracle allowed),
and, with the formal token, how many scored tries are left and whether Problem 1
is still open.

To switch to formal runs, set `EXAMINER_TOKEN` to your formal token in the top
configuration cell, then rerun that cell and this connection cell. Check that
the output says `token: exam` and shows the remaining tries. Editing the token
without rerunning the cells does not update the client; exporting it in another
terminal does not change the running Jupyter kernel.


In [ ]:
from test_client import ExaminerClient, ModelClient, RemoteError

examiner = ExaminerClient(EXAMINER_URL, EXAMINER_TOKEN)
spec = examiner.spec()
print("token:", spec["mode"], "| costs:", spec["costs"], "| budget per case:", spec["budget"],
      "| tool limit:", spec["tool_limit"], "| oracle limit:", spec["oracle_limit"])
print("test cases:", len(spec["test_ids"]), "| interactive:", len(spec["interactive_ids"]),
      "| frozen initial answers:", spec["frozen_initials"])
if spec["mode"] == "trial":
    print("trial quota on test cases:", spec["trial_quota"])
else:
    print("tries left:", spec["submissions_left"], "of", spec["max_submissions"],
          "| Problem 1 closed at:", spec["problem1_closed_at"])

model = ModelClient(MODEL_URL, MODEL_TOKEN)
print("model:", model.model_id)

## 2. The data

Labeled cases are in `data/train/` (splits `train`, `cal_fit`, `cal_select`,
`dev`); the test inputs are in `data/test/cases.jsonl`, and the examiner's frozen
initial answers for them in `data/test/initial_M9.jsonl`. These are model answers,
not test labels.

The 660 labeled cases already include Qwen initial answers and trajectories.
This notebook demonstrates collecting them yourself. To use the provided records
instead, see the [optional offline path in the README](../README.md#可选直接使用老师提供的初始作答).
They cover only the initial stage; section 5 collects the post-query trajectories
needed to fit post-query risks.


In [ ]:
from collections import Counter
from baselines.runner import LABELED_SPLITS, labeled_cases, load_cases, load_frozen

labeled = load_cases(DATA)
test_cases = load_cases(DATA, ["test"])
frozen = load_frozen(DATA)
print(Counter((c["training_split"], c["domain"]) for c in labeled))
print("test cases:", len(test_cases), "| frozen initial answers:", len(frozen))

example = {d: next(c for c in labeled if c["domain"] == d and c["training_split"] == "train")
           for d in spec["domains"]}
for domain, case in example.items():
    print(f"\n{domain}: inputs {sorted(case['inputs'])}\n  label {case['label']}")

## 3. Tools and the oracle by hand

An *episode* is one attempt at one case. Labeled cases get a fresh episode each
time you open them; each test case has one episode per team that keeps all calls
and costs. A tool call costs 0.01 (0.03 with `mode="independent"` on noisy tools);
the nth new oracle read of a case costs 0.10 * n (0.10, 0.20, 0.30, ...).
Reading the same field in the same phase again is free and does not advance n.
The per-case budget is still 2.00; view.oracle_pricing gives the next price.

In [ ]:
def input_tool(episode, name, mode="default"):
    """Call a tool whose required arguments all come from the case inputs."""
    tool = next(t for t in episode["view"]["tools"] if t["name"] == name)
    required = tool["inputSchema"]["json"].get("required", [])
    args = {k: episode["view"]["inputs"][k] for k in required}
    return examiner.tool(episode["episode_id"], name, args, mode)["receipt"]

dg = examiner.episode(example["dangerous_goods"]["case_id"])
print("tools:", [t["name"] for t in dg["view"]["tools"]])
print(input_tool(dg, "calculate_sds_label_score"))
print(input_tool(dg, "calculate_sds_label_score", mode="independent"))   # second source
print(examiner.oracle(dg["episode_id"], "sds", "assessment")["receipt"])   # the truth
history = examiner.history(dg["episode_id"])
print("cost so far:", history["cost"], "| tool calls:", history["tool_calls"],
      "| oracle calls:", history["oracle_calls"])

In [ ]:
air = examiner.episode(example["aircraft_inspection"]["case_id"])
print(input_tool(air, "VerifyAircraftClearance"))

cs = examiner.episode(example["customer_service"]["case_id"])
print(input_tool(cs, "validateAccount"))
try:   # customer KPIs exist only after diagnostics, so this is refused
    examiner.oracle(cs["episode_id"], "latency", "pre_repair")
except RemoteError as error:
    print("refused:", error)

## 4. One baseline episode with Qwen

The agent reads the SOP, calls tools one at a time and ends with a
`final_proposal` that includes a self-reported error probability. Four extra
tool-free samples from the same evidence give the `disagreement` risk.

In [ ]:
from baselines.agent import INITIAL_CALL_LIMIT, main_loop, new_snapshot, shared_samples
from baselines.uq import disagreement

case = example["dangerous_goods"]
snapshot = new_snapshot(examiner.episode(case["case_id"]), model)
main_loop(snapshot, model, examiner, limit=INITIAL_CALL_LIMIT)
shared_samples(snapshot, model)
print("answer:", snapshot["answer"], "| label:", case["label"])
print("self-reported risk:", snapshot["p_error"], "| disagreement:", disagreement(snapshot))
tools = [e["receipt"]["tool"] for e in snapshot["history"] if e["kind"] == "tool_receipt"]
print("tool calls:", tools, "| cost:", snapshot["cost"])

## 5. Collect labeled trajectories and fit risk estimators

After the configuration, connection and data cells (sections 1 and 2), run every
code cell in this section to prepare `calibrator`, `METHOD` and the result-reading
imports for formal baseline runs. Sections 3 and 4 are hands-on examples and can
be skipped when returning to this workflow.

One pass collects everything: `run_cases` with the `fixed` policy runs the agent
on each labeled case (initial answer and four samples), reads one oracle fact
and lets the agent continue. `initial.jsonl` then trains the initial risk and
`posterior.jsonl` the two post-query risks. Progress is saved after every step,
so re-running the cell with the same configuration resumes.

`LIMIT` cases per split keeps this first pass short. To complete the baseline
collection, set `LIMIT = None` and rerun to extend it to all 660 cases, then fit
again. A labeled case takes about 10 seconds on one GPU on average (customer-service
cases about 15 seconds), so all 660 cases take about 2 hours; start that early.
`fit` uses `train` for the historical and supervised estimators, `cal_fit` for
calibration and `cal_select` for the DEFER thresholds.

The collection call below uses `policy="fixed", fixed_count=1`. Changing the
later `METHOD` alone does not change this collection call. If you change the query
policy for test runs, collect matching labeled query trajectories and refit the
post-query risks as well.


In [ ]:
from baselines.runner import run_cases
from baselines.storage import read_jsonl
from baselines.uq import Calibrator

LIMIT = 4   # cases per split for a first pass; None = all 660
cases = labeled_cases(DATA, LABELED_SPLITS, LIMIT)
print(run_cases(examiner, cases, WORK / "collect", model=model, policy="fixed", fixed_count=1))

initial = read_jsonl(WORK / "collect" / "initial.jsonl")
posterior = read_jsonl(WORK / "collect" / "posterior.jsonl")
calibrator = Calibrator().fit(labeled, initial)
calibrator.select_thresholds(labeled, initial, split="cal_select")
calibrator.fit_posterior(labeled, posterior)
try:
    calibrator.select_thresholds(labeled, posterior, split="cal_select", posterior=True)
except ValueError:
    print("No queried cal_select case yet; the initial thresholds apply after queries.")
calibrator.save(WORK / "calibrator.json")
print(calibrator.fit_counts)
print({method: round(record["threshold"], 3) for method, record in calibrator.selections.items()})

### The two post-query risks

After an oracle read the agent continues. Its new self-reports estimate
`p_initial_error_after_queries` (is the *initial* answer wrong?) and
`p_final_error` (is the *final* answer wrong?); `fit_posterior` above calibrated
both on the labeled query trajectories. The test run below must use the same
query policy, so `METHOD` keeps it in one place.

In [ ]:
METHOD = dict(uq="calibrated_disagreement", policy="fixed", fixed_count=1)

### Score labeled results locally

Every run writes `results.csv`. On labeled cases you can score it yourself with
the same rules the examiner uses; this costs no tries.

In [ ]:
from test_client import read_results, score
from test_client.results import check_results

def local_score(path):
    index = {c["case_id"]: c for c in labeled}
    rows = check_results(read_results(path), index, set(index), require_all=False)
    return score(rows, {cid: index[cid] for cid in rows}, set(index))

report = local_score(WORK / "collect" / "results.csv")
for section in ("initial", "updated_initial", "final"):
    print(section, report[section]["domain_macro"])
print("decision", report["decision"]["domain_macro"])

## 6. Problem 1: test-case risks, then submit

The examiner fixed one initial answer per test case (same for every team). This
run reads no oracle: it estimates `p_initial_error` on all 360 cases and decides
`AUTO` or `DEFER` on the 72 interactive cases from that risk alone, without model
calls. **The cell below generates and submits a CSV.** With the formal token each
valid upload uses one try and is scored at once; a file that fails the check is
rejected, and an identical re-upload returns its earlier report, neither using a
try. With the trial token the file is only checked. Use the feedback to judge your
method, not to infer test labels. `scripts/run_test.py` submits from a terminal.

After switching to your formal token and rerunning sections 1, 2 and 5, run this
section before section 7: it sets `RUN` for the active identity. Trial results go
to `WORK / "trial"`, formal results to `WORK / "exam"`. To try a new method without
reusing earlier run files, set a new directory here, such as
`RUN = WORK / spec["mode"] / "v2"`; sections 7 and 8 use the same `RUN`.


In [ ]:
RUN = WORK / spec["mode"]                  # trial and formal test runs stay apart
problem1 = run_cases(examiner, test_cases, RUN / "problem1", model=model, frozen=frozen,
                     calibrator=calibrator, **dict(METHOD, policy="threshold"))
print(problem1)

def show(answer):
    if answer.get("trial"):
        print("trial token: the file is valid; not scored, no try used")
        return
    print("try", answer["submission"], "| tries left:", answer["submissions_left"],
          "| raw-metric best:", answer["best"])
    for section in ("initial", "updated_initial", "final", "decision"):
        print(section, answer["report"][section]["domain_macro"])

results = RUN / "problem1" / "results.csv"
test_index = {c["case_id"]: c for c in test_cases}
check_results(read_results(results), test_index, set(spec["interactive_ids"]))
show(examiner.submit(results))             # formal token: uses one try

## 7. Problem 2: oracle reads and decisions, then submit

Now the runner reads one oracle fact on each interactive case, lets the agent
continue, estimates both post-query risks and decides `AUTO` or `DEFER`, then
submits the CSV. The examiner refuses test-case oracle reads until you have
submitted once. Your first successful formal test oracle read closes Problem 1:
p no longer updates; only q/r and decision grades update.

Finish your Problem 1 improvements and submissions before setting
`DONE_WITH_PROBLEM_1 = True`. If you want to use the custom p example in section 8,
run and submit it before this section. With the trial token, this section runs
within its small quota without changing the flag.

The original `best.problem2` field records the lowest decision loss. The
examiner's `report.grades` combines this try's q/r and decision grades with
the best eligible pre-oracle p grade. These effective submission grades go
to the big screen: its detail table shows the latest try, while its home
ranking keeps the highest total. `report.submission_grades` preserves this
CSV's standalone grades, including its unadjusted p grade, for analysis.
See the [candidate manual](../docs/candidate_manual.pdf) for the full grading rule.


In [ ]:
DONE_WITH_PROBLEM_1 = False   # formal token: True once your Problem 1 tries are done

if spec["mode"] == "trial" or DONE_WITH_PROBLEM_1:
    problem2 = run_cases(examiner, test_cases, RUN / "problem2", model=model, frozen=frozen,
                         calibrator=calibrator, **METHOD)
    print(problem2)
    show(examiner.submit(RUN / "problem2" / "results.csv"))
else:
    print("Skipped this Problem 2 run; see the closing time below for Problem 1 status.")
listing = examiner.submissions()
for item in listing["submissions"]:
    print("try", item["submission"], "| p_i Brier:", item["primary"])
print("raw-metric best:", listing["best"], "| Problem 1 closed at:", listing["problem1_closed_at"])

## 8. Your own method

Any method works as long as it writes the same `results.csv`: one row per test
case with `p_initial_error`; on interactive cases also `mode`, the final answer,
`p_initial_error_after_queries` and `p_final_error`. The initial answer must be
the frozen one, and `p_initial_error` counts only in tries made before Problem 1
closes. Example: shrink the Problem 1 risks toward each domain's historical error
rate, then check the file before submitting.

The cell below writes and checks `my_results.csv`; it does not submit it. To use
these p estimates, run this example after section 6 and submit the file before
section 7. From the project root, in a terminal configured with the intended examiner
URL and token, run
`python scripts/run_test.py` followed by the file path printed below. For the
default formal-run directory, this is:

```bash
python scripts/run_test.py .runs/notebook/exam/my_results.csv
```


In [ ]:
from test_client import write_results

rows = read_results(RUN / "problem1" / "results.csv")
domain_of = {c["case_id"]: c["domain"] for c in test_cases}
for case_id, row in rows.items():
    prior = calibrator.rates[domain_of[case_id]]
    if row["p_initial_error"] is not None:
        row["p_initial_error"] = 0.7 * row["p_initial_error"] + 0.3 * prior
mine = write_results(RUN / "my_results.csv", [{"case_id": k, **v} for k, v in rows.items()])
check_results(read_results(mine), test_index, set(spec["interactive_ids"]))
print("ready to submit:", mine)